# Preparacion DB y Creacion indices para Análisis de la Encuesta de Competencias Financieras 2021

## Librerias

In [88]:
import pandas as pd
import numpy as np
from pathlib import Path


## Carga del fichero csv

In [89]:
df = pd.read_csv(
    r"..\Data\Silver_ECF_090226.csv",
    sep=","
)
df.shape

(7554, 55)

In [90]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7554 entries, 0 to 7553
Data columns (total 55 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   cat_educacion          7554 non-null   object 
 1   estado_laboral         7554 non-null   object 
 2   familia                7554 non-null   object 
 3   riq_inmob              7554 non-null   object 
 4   ahorro                 7554 non-null   int64  
 5   deuda                  7554 non-null   int64  
 6   adq_ahorro2            7554 non-null   int64  
 7   adq_deuda2             7554 non-null   int64  
 8   adq_otro               7554 non-null   int64  
 9   ahorra_12m             7554 non-null   int64  
 10  consumo_ahorros        7554 non-null   int64  
 11  cred_inf               7554 non-null   int64  
 12  cred_form              7554 non-null   int64  
 13  cred_add               7554 non-null   int64  
 14  impago                 7554 non-null   int64  
 15  ayud

In [91]:
df.head()

,cat_educacion,estado_laboral,familia,riq_inmob,ahorro,deuda,adq_ahorro2,adq_deuda2,adq_otro,ahorra_12m,...,venta_activos,prestamo_fam,prestamo_amig,tarjeta_cred,descubierto_auto,prest_garantia,linea_cred,prest_banc,prest_nobanc,otro_prest
0,Universidad,Asalariado,Vive con pareja,Solo residencia principal,0,1,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0
1,Universidad,Asalariado,Otros adultos,Residencia principal y otras,0,1,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
2,Educación Secundaria,Asalariado,Vive solo,Residencia principal y otras,1,0,1,0,1,1,...,-98,-98,-98,-98,-98,-98,-98,-98,-98,-98
3,Educación Secundaria,Asalariado,Vive con pareja,Solo residencia principal,1,1,0,0,0,1,...,-98,-98,-98,-98,-98,-98,-98,-98,-98,-98
4,Educación Primaria,Asalariado,Otros adultos,Solo residencia principal,0,1,0,1,1,0,...,0,0,0,0,0,0,0,1,0,0


In [92]:
df.columns.tolist()

['cat_educacion',
 'estado_laboral',
 'familia',
 'riq_inmob',
 'ahorro',
 'deuda',
 'adq_ahorro2',
 'adq_deuda2',
 'adq_otro',
 'ahorra_12m',
 'consumo_ahorros',
 'cred_inf',
 'cred_form',
 'cred_add',
 'impago',
 'ayudas',
 'rechazo',
 'semirechazo',
 'nopiden',
 'age',
 'ID',
 'year',
 'inflacion_ok',
 'interes_comp_ok',
 'diversificacion_ok',
 'weight',
 't_hipoteca',
 't_ppension_ind',
 't_ppension_emp',
 't_fondo_inv',
 't_acciones',
 't_prestamo',
 't_renta_fija',
 't_cuentas_depositos',
 'adq_hipoteca',
 'adq_ppension_ind',
 'adq_ppension_emp',
 'adq_fondo_inv',
 'adq_acciones',
 'adq_prest',
 'adq_otros_act',
 'adq_renta_fija',
 'adq_cuentas_depositos',
 'uso_cc',
 'uso_ahorro_dep',
 'venta_activos',
 'prestamo_fam',
 'prestamo_amig',
 'tarjeta_cred',
 'descubierto_auto',
 'prest_garantia',
 'linea_cred',
 'prest_banc',
 'prest_nobanc',
 'otro_prest']

## 1. Transformación del dataset (Gold)

### 1.1. Creación de grupos de edad

In [93]:
def make_age_group(s: pd.Series) -> pd.Categorical:
    bins = [18, 35, 45, 55, 65, 81]
    labels = ["18-34 años", "35-44 años", "45-54 años", "55-64 años", "65-79 años"]
    return pd.cut(s, bins=bins, right=False, labels=labels, include_lowest=True)

### 1.2 Limpieza inicial y creación de edad_grupo

- `age` a formato numérico
- Creación `edad_grupo` variable categórica 

In [94]:
df = df.copy()
df.columns = df.columns.str.strip()

df["age"] = pd.to_numeric(df["age"], errors="coerce")
df["edad_grupo"] = make_age_group(df["age"])


#### 1.3 Cálculo del score de competencia financiera (0–3)


puntuación agregada a partir de tres variables:

- `inflacion_ok`
- `interes_comp_ok`
- `diversificacion_ok`

Transformaciones:

- La puntuación final (`score_comp_fin`) es la suma de las tres variables, El score resultante toma valores entre 0 y 3.

In [95]:
cols_comp = ["inflacion_ok", "interes_comp_ok", "diversificacion_ok"]

df["score_comp_fin"] = (
    df[cols_comp]
    .replace(-1, 0)
    .sum(axis=1)
)


#### 1.4 Creación del indicador categórico de competencia financiera

A partir de `score_comp_fin` se construye `nivel_comp_fin`, clasifica en tres categorías:

- 0–1 puntos → bajo  
- 2 puntos → medio  
- 3 puntos → alto  

permite segmentar la muestra según nivel de competencia financiera.


In [96]:
def categorizar_competencia(score):
    if score <= 1:
        return "bajo"
    elif score == 2:
        return "medio"
    else:  # score == 3
        return "alto"

df["nivel_comp_fin"] = df["score_comp_fin"].apply(categorizar_competencia)

In [97]:
# Vemos como queda distribuido por categorias
df["score_comp_fin"].value_counts().sort_index()
df["nivel_comp_fin"].value_counts()

nivel_comp_fin
bajo     3342
medio    2646
alto     1566
Name: count, dtype: int64

In [98]:
df["nivel_comp_fin"].value_counts(normalize=True)

nivel_comp_fin
bajo     0.442415
medio    0.350278
alto     0.207307
Name: proportion, dtype: float64

#### 1.5 Creación del indicador binario de competencia financiera

- 1 → competente (score ≥ 2)
- 0 → no competente (score ≤ 1)


In [99]:
df["comp_fin_bin"] = (df["score_comp_fin"] >= 2).astype(int)

#### 1.6 Construcción del indicador de vulnerabilidad

`vulnerabilidad`

- Uso de préstamos para cubrir gastos
- Morosidad
- Denegación de crédito
- Uso de ahorros para cubrir gastos

La variable toma valor 1 si se cumple al menos una de las condiciones anteriores.


In [100]:
# VARIABLES DE VULNERABILIDAD

# 1. Uso de préstamos para cubrir gastos
df["prestamo_gastos"] = (
    (df["cred_inf"] == 1) |
    (df["cred_form"] == 1) |
    (df["cred_add"] == 1)
).astype(int)

# 2. Morosidad
df["morosidad"] = (df["impago"] == 1).astype(int)

# 3. Denegación de crédito
df["denegacion_credito"] = (
    (df["rechazo"] == 1) |
    (df["semirechazo"] == 1)
).astype(int)

# 4. Uso de ahorros para cubrir gastos
df["uso_ahorros"] = (df["consumo_ahorros"] == 1).astype(int)


# CÁLCULO DEL INDICADOR DE VULNERABILIDAD (1 si cumple al menos una condición)

df["vulnerabilidad"] = (
    (df["prestamo_gastos"] == 1) |
    (df["morosidad"] == 1) |
    (df["denegacion_credito"] == 1) |
    (df["uso_ahorros"] == 1)
).astype(int)


# CHECK

df[[
    "prestamo_gastos",
    "morosidad",
    "denegacion_credito",
    "uso_ahorros",
    "vulnerabilidad"
]].mean().round(3)

prestamo_gastos       0.101
morosidad             0.027
denegacion_credito    0.044
uso_ahorros           0.132
vulnerabilidad        0.235
dtype: float64

In [101]:
resumen_vuln = pd.DataFrame({
    "conteo": df["vulnerabilidad"].value_counts().sort_index(),
    "proporcion": df["vulnerabilidad"].value_counts(normalize=True).sort_index()
})

resumen_vuln.round(3)


,conteo,proporcion
vulnerabilidad,,
0,5780,0.765
1,1774,0.235


In [102]:
df["vulnerabilidad"].describe()

count    7554.000000
mean        0.234842
std         0.423928
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max         1.000000
Name: vulnerabilidad, dtype: float64

#### 1.7 Creación del índice de estrés financiero

`idx_estres_financiero` suma de las siguientes variables:
- `consumo_ahorros`
- `cred_inf`
- `cred_form`
- `cred_add`
- `impago`
- `ayudas`


In [103]:
stress_cols = [
    "consumo_ahorros",
    "cred_inf",
    "cred_form",
    "cred_add",
    "impago",
    "ayudas"
]

df["idx_estres_financiero"] = df[stress_cols].sum(axis=1)


#### 1.8 Creación del índice de uso de crédito

Se construye el índice `idx_uso_credito` como suma de:

- `deuda`
- `adq_deuda2`
- `cred_form`
- `cred_add`


In [104]:
credit_cols = [
    "deuda",
    "adq_deuda2",
    "cred_form",
    "cred_add"
]

df["idx_uso_credito"] = df[credit_cols].sum(axis=1)


#### 1.9 Creación del índice de ahorro

`idx_ahorro` como suma de:

- `ahorro`
- `adq_ahorro2`
- `ahorra_12m`


In [105]:
save_cols = [
    "ahorro",
    "adq_ahorro2",
    "ahorra_12m"
]

df["idx_ahorro"] = df[save_cols].sum(axis=1)


#### 1.10 adjuntar y transformar columnas de productos


In [106]:
df = df.rename(columns={"adq_prest": "adq_prestamo"})


In [107]:
def tobinary01(x) -> float:

    if pd.isna(x):
        return 0.0

    # booleans
    if isinstance(x, (bool, np.bool_)):
        return float(int(x))

    # numbers
    if isinstance(x, (int, np.integer, float, np.floating)):

        # Valores válidos
        if x == 1:
            return 1.0
        if x == 0:
            return 0.0
        if x == 2:
            return 0.0

        # Códigos especiales → convertir a 0
        if x in (-97, -98, -99, -5):
            return 0.0

        return 0.0

    # strings
    s = str(x).strip().lower()
    yes = {"si", "sí", "yes", "y", "true", "t", "1"}
    no  = {"no", "n", "false", "f", "0"}

    if s in yes:
        return 1.0
    if s in no:
        return 0.0

    return 0.0


In [109]:
# Forzar limpieza final definitiva

for c in product_cols:
    if c in df.columns:
        df[c] = df[c].fillna(0)      # NaN → 0
        df[c] = df[c].astype(int)    # float → int


In [110]:
def crear_df_brechas(df):


    # === 2. Definir grupos de productos ===

    ahorro_cols = [
        "t_cuentas_depositos", "adq_cuentas_depositos",
        "uso_cc", "uso_ahorro_dep"
    ]

    inversion_cols = [
        "t_fondo_inv", "t_acciones", "t_renta_fija",
        "t_ppension_ind", "t_ppension_emp",
        "adq_fondo_inv", "adq_acciones", "adq_renta_fija",
        "adq_ppension_ind", "adq_ppension_emp"
    ]

    credito_cols = [
        "t_hipoteca", "t_prestamo",
        "adq_hipoteca", "adq_prestamo",
        "prestamo_fam", "prestamo_amig",
        "tarjeta_cred", "descubierto_auto",
        "prest_garantia", "linea_cred",
        "prest_banc", "prest_nobanc", "otro_prest"
    ]

    # === 3. Crear scores por categoría ===
    df["score_ahorro"] = df[ahorro_cols].sum(axis=1)
    df["score_inversion"] = df[inversion_cols].sum(axis=1)
    df["score_credito"] = df[credito_cols].sum(axis=1)

    # === 4. Actividad financiera total ===
    df["actividad_score"] = (
        df["score_ahorro"] +
        df["score_inversion"] +
        df["score_credito"]
    )

    df["actividad"] = (df["actividad_score"] > 1).astype(int)

    # === 5. Complejidad de productos ===
    # Inversión + crédito = productos más complejos
    df["complejidad"] = (
        (df["score_inversion"] > 0) |
        (df["score_credito"] > 0)
    ).astype(int)

    # === 6. Competencia de producto === Entendido no como conocimiento financiero (eso seria comp_fin_bin),
    # sino como medida de uso real de productos financieros, es decir: ¿La persona interactúa con productos financieros? ¿Tiene experiencia práctica?
    df["comp_prod"] = (df["actividad_score"] > 0).astype(int)



    return df

In [113]:
# Aplicar función y crear nuevas columnas en el dataframe
df = crear_df_brechas(df)

# Verificar que las nuevas variables existen
df[[
    "score_ahorro",
    "score_inversion",
    "score_credito",
    "actividad_score",
    "actividad",
    "complejidad",
    "comp_prod"
]].head()


,score_ahorro,score_inversion,score_credito,actividad_score,actividad,complejidad,comp_prod
0,1,0,3,4,1,1,1
1,0,0,2,2,1,1,1
2,2,7,0,9,1,1,1
3,0,2,1,3,1,1,1
4,0,0,3,3,1,1,1


In [114]:
# Lista completa colonne prodotto binarizzate
product_cols = [
    't_hipoteca','t_ppension_ind','t_ppension_emp','t_fondo_inv','t_acciones',
    't_prestamo','t_renta_fija','t_cuentas_depositos',
    'adq_hipoteca','adq_ppension_ind','adq_ppension_emp',
    'adq_fondo_inv','adq_acciones','adq_prestamo','adq_otros_act',
    'adq_renta_fija','adq_cuentas_depositos',
    'uso_cc','uso_ahorro_dep','venta_activos',
    'prestamo_fam','prestamo_amig','tarjeta_cred',
    'descubierto_auto','prest_garantia','linea_cred',
    'prest_banc','prest_nobanc','otro_prest'
]

print("=== CHECK BINARIZACIÓN ===")

for c in product_cols:
    if c in df.columns:
        unique_vals = sorted(df[c].unique())
        invalid = [v for v in unique_vals if v not in [0,1]]
        
        print(f"\n{c}")
        print("Valores únicos:", unique_vals)
        
        if len(invalid) == 0:
            print("OK → Solo 0/1")
        else:
            print("ERROR → Valores no binarios detectados:", invalid)

print("\n=== FIN CHECK ===")


=== CHECK BINARIZACIÓN ===

t_hipoteca
Valores únicos: [0, 1]
OK → Solo 0/1

t_ppension_ind
Valores únicos: [0, 1]
OK → Solo 0/1

t_ppension_emp
Valores únicos: [0, 1]
OK → Solo 0/1

t_fondo_inv
Valores únicos: [0, 1]
OK → Solo 0/1

t_acciones
Valores únicos: [0, 1]
OK → Solo 0/1

t_prestamo
Valores únicos: [0, 1]
OK → Solo 0/1

t_renta_fija
Valores únicos: [0, 1]
OK → Solo 0/1

t_cuentas_depositos
Valores únicos: [0, 1]
OK → Solo 0/1

adq_hipoteca
Valores únicos: [0, 1]
OK → Solo 0/1

adq_ppension_ind
Valores únicos: [0, 1]
OK → Solo 0/1

adq_ppension_emp
Valores únicos: [0, 1]
OK → Solo 0/1

adq_fondo_inv
Valores únicos: [0, 1]
OK → Solo 0/1

adq_acciones
Valores únicos: [0, 1]
OK → Solo 0/1

adq_prestamo
Valores únicos: [0, 1]
OK → Solo 0/1

adq_otros_act
Valores únicos: [0, 1]
OK → Solo 0/1

adq_renta_fija
Valores únicos: [0, 1]
OK → Solo 0/1

adq_cuentas_depositos
Valores únicos: [0, 1]
OK → Solo 0/1

uso_cc
Valores únicos: [0, 1]
OK → Solo 0/1

uso_ahorro_dep
Valores únicos: [0, 

### 2. Check finales y Export GOLD df

In [115]:
df.columns

Index(['cat_educacion', 'estado_laboral', 'familia', 'riq_inmob', 'ahorro',
       'deuda', 'adq_ahorro2', 'adq_deuda2', 'adq_otro', 'ahorra_12m',
       'consumo_ahorros', 'cred_inf', 'cred_form', 'cred_add', 'impago',
       'ayudas', 'rechazo', 'semirechazo', 'nopiden', 'age', 'ID', 'year',
       'inflacion_ok', 'interes_comp_ok', 'diversificacion_ok', 'weight',
       't_hipoteca', 't_ppension_ind', 't_ppension_emp', 't_fondo_inv',
       't_acciones', 't_prestamo', 't_renta_fija', 't_cuentas_depositos',
       'adq_hipoteca', 'adq_ppension_ind', 'adq_ppension_emp', 'adq_fondo_inv',
       'adq_acciones', 'adq_prestamo', 'adq_otros_act', 'adq_renta_fija',
       'adq_cuentas_depositos', 'uso_cc', 'uso_ahorro_dep', 'venta_activos',
       'prestamo_fam', 'prestamo_amig', 'tarjeta_cred', 'descubierto_auto',
       'prest_garantia', 'linea_cred', 'prest_banc', 'prest_nobanc',
       'otro_prest', 'edad_grupo', 'score_comp_fin', 'nivel_comp_fin',
       'comp_fin_bin', 'prestamo_gasto

In [116]:
df.head()

,cat_educacion,estado_laboral,familia,riq_inmob,ahorro,deuda,adq_ahorro2,adq_deuda2,adq_otro,ahorra_12m,...,idx_estres_financiero,idx_uso_credito,idx_ahorro,score_ahorro,score_inversion,score_credito,actividad_score,actividad,complejidad,comp_prod
0,Universidad,Asalariado,Vive con pareja,Solo residencia principal,0,1,0,1,1,1,...,1,2,1,1,0,3,4,1,1,1
1,Universidad,Asalariado,Otros adultos,Residencia principal y otras,0,1,0,0,0,0,...,1,1,0,0,0,2,2,1,1,1
2,Educación Secundaria,Asalariado,Vive solo,Residencia principal y otras,1,0,1,0,1,1,...,0,0,3,2,7,0,9,1,1,1
3,Educación Secundaria,Asalariado,Vive con pareja,Solo residencia principal,1,1,0,0,0,1,...,0,1,2,0,2,1,3,1,1,1
4,Educación Primaria,Asalariado,Otros adultos,Solo residencia principal,0,1,0,1,1,0,...,1,3,0,0,0,3,3,1,1,1


In [117]:
# Revisión final del dataset GOLD

CHECK_COLS = [
    "score_comp_fin",
    "nivel_comp_fin",
    "comp_fin_bin",
    "vulnerabilidad",
    "idx_estres_financiero",
    "idx_uso_credito",
    "idx_ahorro",
    "edad_grupo",
    "weight"
]

print("Columnas clave no encontradas en el dataset:")
print([c for c in CHECK_COLS if c not in df.columns])

print("\nValores perdidos en variables principales:")
print(df[CHECK_COLS].isna().sum())

print("\nResumen estadístico del score de competencia financiera (esperado 0–3):")
print(df["score_comp_fin"].describe())

print("\nResumen estadístico de los índices compuestos:")
print(df[[
    "idx_estres_financiero",
    "idx_uso_credito",
    "idx_ahorro"
]].describe())

print("\nDistribución del indicador binario de competencia financiera:")
print(df["comp_fin_bin"].value_counts(normalize=True).round(3))

print("\nDistribución del indicador de vulnerabilidad:")
print(df["vulnerabilidad"].value_counts(normalize=True).round(3))

print("\nRevisión completada.")


Columnas clave no encontradas en el dataset:
[]

Valores perdidos en variables principales:
score_comp_fin           0
nivel_comp_fin           0
comp_fin_bin             0
vulnerabilidad           0
idx_estres_financiero    0
idx_uso_credito          0
idx_ahorro               0
edad_grupo               0
weight                   0
dtype: int64

Resumen estadístico del score de competencia financiera (esperado 0–3):
count    7554.000000
mean        1.641250
std         0.944493
min         0.000000
25%         1.000000
50%         2.000000
75%         2.000000
max         3.000000
Name: score_comp_fin, dtype: float64

Resumen estadístico de los índices compuestos:
       idx_estres_financiero  idx_uso_credito   idx_ahorro
count            7554.000000      7554.000000  7554.000000
mean                0.283823         0.667990     1.389727
std                 0.604003         0.841097     0.995065
min                 0.000000         0.000000     0.000000
25%                 0.000000   

In [120]:
df.to_csv(
    r"..\Data\Gold_ECF_090226.csv",
    index=False
)
